# NB19 — CFTR Transfer: MASTER vs MASTER+KANSER+PAH

**TEKNOFEST Sağlıkta Yapay Zeka | Genetik Varyant Patojenite Tahmini**

## Amaç

NB17 CFTR panel refinement'ında **S0_MASTER-only** en iyi strateji olarak çıktı (LOO-MCC=0.627, prior-shift ile 0.655). Soru: **Eğitim havuzunu MASTER'dan MASTER+KANSER+PAH'a (birleşik 3691 satır) genişletmek CFTR transferi iyileştirir mi?**

Bu notebook:
1. MASTER (2931) + KANSER (388) + PAH (372) → COMBINED (3691 satır) oluştur
2. CFTR'den birebir-aynı satırları drop et
3. NB17 S0 protokolü uygula: LightGBM + LOO-CV + MCC + prior-shift
4. **S0_MASTER-only vs S0c_Combined** karşılaştırma

Eğitim havuzunun %74'ü pathogenic olan bu senaryo, CFTR'ye transfer yazısını değiştirir mi?

## Cell 1: Imports & Config

In [1]:
import os, sys, warnings, json
from copy import deepcopy
from datetime import datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split, LeaveOneOut, cross_val_predict
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    matthews_corrcoef, confusion_matrix
)
from sklearn.pipeline import Pipeline
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(PROJECT_ROOT, "config.py")):
    PROJECT_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, REPORTS_DIR
from src import columns_real as CR
import lightgbm as lgb

# LightGBM sabit parametreleri (src.models'den kopya, torch dependency'si olmadan)
LGBM_FIXED = {
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'verbosity': -1,
    'random_state': SEED,
    'objective': 'binary',
    'class_weight': 'balanced',
}

np.random.seed(SEED)

# --- Sabitler ---
PANEL             = "CFTR"
HIGH_MISSING_THR  = 0.50
FINAL_BENIGN_FRAC = 0.80
N_BOOT            = 50
BOOT_SEED         = SEED
PI_TEST           = 0.20  # final test orani

DATA_DIR    = os.path.join(PROJECT_ROOT, "data", "real_data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v9_cftr_combined")
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}  SEED: {SEED}")
print(f"RESULTS_DIR: {RESULTS_DIR}")

PROJECT_ROOT: /Users/tefe/teknofest_model/teknofest_model  SEED: 42
RESULTS_DIR: /Users/tefe/teknofest_model/teknofest_model/results/v9_cftr_combined


## Cell 2: Veri Yükleme + Sütun Temizliği

In [2]:
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

def load_panel(name):
    return pd.read_csv(os.path.join(DATA_DIR, CR.PANEL_INFO[name]["file"]))

master = load_panel("MASTER")
kanser = load_panel("KANSER")
pah    = load_panel("PAH")
cftr_raw = load_panel("CFTR")

feature_cols_all = [c for c in master.columns if c not in (ID_COL, TARGET)]

def drop_exact_duplicates(df, ref_df, name):
    """df'den ref_df ile birebir-aynı satırları drop et."""
    ref_index = {}
    for _, row in ref_df.iterrows():
        key = (row[ID_COL],) + tuple((np.nan if pd.isna(v) else v) for v in row[feature_cols_all].values)
        ref_index[key] = row[TARGET]
    
    drop_idx = [idx for idx, row in df.iterrows()
                if (((row[ID_COL],) + tuple((np.nan if pd.isna(v) else v)
                     for v in row[feature_cols_all].values)) in ref_index
                    and ref_index[(row[ID_COL],) + tuple((np.nan if pd.isna(v) else v)
                         for v in row[feature_cols_all].values)] == row[TARGET])]
    
    df_clean = df.drop(index=drop_idx).reset_index(drop=True)
    print(f"{name}: {df.shape[0]} -> {df_clean.shape[0]} (drop={len(drop_idx)} birebir-aynı satır)")
    return df_clean

# CFTR'den MASTER ile birebir-aynı satırları drop et
cftr = drop_exact_duplicates(cftr_raw, master, "CFTR")

# KANSER ve PAH'tan CFTR ile birebir-aynı satırları drop et
kanser_clean = drop_exact_duplicates(kanser, cftr_raw, "KANSER")
pah_clean    = drop_exact_duplicates(pah, cftr_raw, "PAH")

print(f"\nMESTER: {master.shape}, label: {master[TARGET].value_counts().to_dict()}")
print(f"KANSER: {kanser_clean.shape}, label: {kanser_clean[TARGET].value_counts().to_dict()}")
print(f"PAH   : {pah_clean.shape}, label: {pah_clean[TARGET].value_counts().to_dict()}")
print(f"CFTR  : {cftr.shape}, label: {cftr[TARGET].value_counts().to_dict()}")

# COMBINED: MASTER + KANSER + PAH
combined = pd.concat([master, kanser_clean, pah_clean], ignore_index=True)
print(f"\nCOMBINED: {combined.shape}, label: {combined[TARGET].value_counts().to_dict()}")

# Sütun temizliği: MASTER üzerinde sabit + özdeş sütun tespiti
constant_cols = CR.get_constant_cols(master[feature_cols_all])
dup_pairs     = CR.get_duplicate_col_pairs(master[feature_cols_all])
dup_drop      = sorted({b for (a, b) in dup_pairs})
drop_cols     = sorted(set(constant_cols) | set(dup_drop))
base_feature_cols = [c for c in feature_cols_all if c not in drop_cols]
CAT_LIKE      = [c for c in (CR.CAT_COLS + CR.AA_COLS) if c in base_feature_cols]
NUM_COLS_BASE = [c for c in base_feature_cols if c not in CAT_LIKE]

print(f"\nSütun temizliği: drop {len(drop_cols)} -> {len(base_feature_cols)} feature")
print(f"  ({len(NUM_COLS_BASE)} sayısal + {len(CAT_LIKE)} kategorik)")

CFTR: 111 -> 111 (drop=0 birebir-aynı satır)
KANSER: 388 -> 388 (drop=0 birebir-aynı satır)


PAH: 372 -> 372 (drop=0 birebir-aynı satır)

MESTER: (2931, 353), label: {1: 2149, 0: 782}
KANSER: (388, 353), label: {1: 268, 0: 120}
PAH   : (372, 353), label: {1: 310, 0: 62}
CFTR  : (111, 353), label: {1: 90, 0: 21}

COMBINED: (3691, 353), label: {1: 2727, 0: 964}



Sütun temizliği: drop 63 -> 288 feature
  (281 sayısal + 7 kategorik)


## Cell 3: FE + M3 Preprocessing

In [3]:
AA_UNK = CR.AA_UNKNOWN_TOKEN
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# Grantham + BLOSUM62 (NB17'den kopyalı)
_GRANTHAM = {
 ('S','R'):110,('S','L'):145,('S','P'):74,('S','T'):58,('S','A'):99,('S','V'):124,
 ('S','G'):56,('S','I'):142,('S','F'):155,('S','Y'):144,('S','C'):112,('S','H'):89,
 ('S','Q'):68,('S','N'):46,('S','K'):121,('S','D'):65,('S','E'):80,('S','M'):135,('S','W'):177,
 ('R','L'):102,('R','P'):103,('R','T'):71,('R','A'):112,('R','V'):96,('R','G'):125,('R','I'):97,
 ('R','F'):97,('R','Y'):77,('R','C'):180,('R','H'):29,('R','Q'):43,('R','N'):86,('R','K'):26,
 ('R','D'):96,('R','E'):54,('R','M'):91,('R','W'):101,
 ('L','P'):98,('L','T'):92,('L','A'):96,('L','V'):32,('L','G'):138,('L','I'):5,('L','F'):22,
 ('L','Y'):36,('L','C'):198,('L','H'):99,('L','Q'):113,('L','N'):153,('L','K'):107,('L','D'):172,
 ('L','E'):138,('L','M'):15,('L','W'):61,
 ('P','T'):38,('P','A'):27,('P','V'):68,('P','G'):42,('P','I'):95,('P','F'):114,('P','Y'):110,
 ('P','C'):169,('P','H'):77,('P','Q'):76,('P','N'):91,('P','K'):103,('P','D'):108,('P','E'):93,
 ('P','M'):87,('P','W'):147,
 ('T','A'):58,('T','V'):69,('T','G'):59,('T','I'):89,('T','F'):103,('T','Y'):92,('T','C'):149,
 ('T','H'):47,('T','Q'):42,('T','N'):65,('T','K'):78,('T','D'):85,('T','E'):65,('T','M'):81,('T','W'):128,
 ('A','V'):64,('A','G'):60,('A','I'):94,('A','F'):113,('A','Y'):112,('A','C'):195,('A','H'):86,
 ('A','Q'):91,('A','N'):111,('A','K'):106,('A','D'):126,('A','E'):107,('A','M'):84,('A','W'):148,
 ('V','G'):109,('V','I'):29,('V','F'):50,('V','Y'):55,('V','C'):192,('V','H'):84,('V','Q'):96,
 ('V','N'):133,('V','K'):97,('V','D'):152,('V','E'):121,('V','M'):21,('V','W'):88,
 ('G','I'):135,('G','F'):153,('G','Y'):147,('G','C'):159,('G','H'):98,('G','Q'):87,('G','N'):80,
 ('G','K'):127,('G','D'):94,('G','E'):98,('G','M'):127,('G','W'):184,
 ('I','F'):21,('I','Y'):33,('I','C'):198,('I','H'):94,('I','Q'):109,('I','N'):149,('I','K'):102,
 ('I','D'):168,('I','E'):134,('I','M'):10,('I','W'):61,
 ('F','Y'):22,('F','C'):205,('F','H'):100,('F','Q'):116,('F','N'):158,('F','K'):102,('F','D'):177,
 ('F','E'):140,('F','M'):28,('F','W'):40,
 ('Y','C'):194,('Y','H'):83,('Y','Q'):99,('Y','N'):143,('Y','K'):85,('Y','D'):160,('Y','E'):122,
 ('Y','M'):36,('Y','W'):37,
 ('C','H'):174,('C','Q'):154,('C','N'):139,('C','K'):202,('C','D'):154,('C','E'):170,('C','M'):196,('C','W'):215,
 ('H','Q'):24,('H','N'):68,('H','K'):32,('H','D'):81,('H','E'):40,('H','M'):87,('H','W'):115,
 ('Q','N'):46,('Q','K'):53,('Q','D'):61,('Q','E'):29,('Q','M'):101,('Q','W'):130,
 ('N','K'):94,('N','D'):23,('N','E'):42,('N','M'):142,('N','W'):174,
 ('K','D'):101,('K','E'):56,('K','M'):95,('K','W'):110,
 ('D','E'):45,('D','M'):160,('D','W'):181,
 ('E','M'):126,('E','W'):152,
 ('M','W'):67,
}
def grantham(a, b):
    if a == b: return 0
    return _GRANTHAM.get((a, b)) or _GRANTHAM.get((b, a)) or -1

_B62_RAW = '''A4 R-1 N-2 D-2 C0 Q-1 E-1 G0 H-2 I-1 L-1 K-1 M-1 F-2 P-1 S1 T0 W-3 Y-2 V0
R5 N0 D-2 C-3 Q1 E0 G-2 H0 I-3 L-2 K2 M-1 F-3 P-2 S-1 T-1 W-3 Y-2 V-3
N6 D1 C-3 Q0 E0 G0 H1 I-3 L-3 K0 M-2 F-3 P-2 S1 T0 W-4 Y-2 V-3
D6 C-3 Q0 E2 G-1 H-1 I-3 L-4 K-1 M-3 F-3 P-1 S0 T-1 W-4 Y-3 V-3
C9 Q-3 E-4 G-3 H-3 I-1 L-1 K-3 M-1 F-2 P-3 S-1 T-1 W-2 Y-2 V-1
Q5 E2 G-2 H0 I-3 L-2 K1 M0 F-3 P-1 S0 T-1 W-2 Y-1 V-2
E5 G-2 H0 I-3 L-3 K1 M-2 F-3 P-1 S0 T-1 W-3 Y-2 V-2
G6 H-2 I-4 L-4 K-2 M-3 F-3 P-2 S0 T-2 W-2 Y-3 V-3
H8 I-3 L-3 K-1 M-2 F-1 P-2 S-1 T-2 W-2 Y2 V-3
I4 L2 K-3 M1 F0 P-3 S-2 T-1 W-3 Y-1 V3
L4 K-2 M2 F0 P-3 S-2 T-1 W-2 Y-1 V1
K5 M-1 F-3 P-1 S0 T-1 W-3 Y-2 V-2
M5 F0 P-2 S-1 T-1 W-1 Y-1 V1
F6 P-4 S-2 T-2 W1 Y3 V-1
P7 S-1 T-1 W-4 Y-3 V-2
S4 T1 W-3 Y-2 V-2
T5 W-2 Y-2 V0
W11 Y2 V-3
Y7 V-1
V4'''
_ORDER = list("ARNDCQEGHILKMFPSTWYV")
_B62 = {}
for _ri, _line in enumerate(_B62_RAW.strip().split("\n")):
    _toks = _line.split()
    _row_aa = _toks[0][0]
    _vals = [_toks[0][1:]] + _toks[1:]
    for _ci, _tok in enumerate(_vals):
        _col_aa = _ORDER[_ri + _ci]
        _v = int(_tok[1:] if _tok[0].isalpha() else _tok)
        _B62[(_row_aa, _col_aa)] = _v; _B62[(_col_aa, _row_aa)] = _v
def blosum62(a, b): return _B62.get((a, b), 0)

def add_fe(df):
    out = df.copy()
    a1 = out["AA_1"].astype("object"); a2 = out["AA_2"].astype("object")
    out["fe_aa_stopgain"] = (a2 == "*").astype(int)
    def _nonstd(v): return 0 if (isinstance(v, str) and v in STANDARD_AA) else 1
    out["fe_aa_nonstandard"] = (a1.map(_nonstd) | a2.map(_nonstd)).astype(int)
    def _gr(r):
        x, y = r["AA_1"], r["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA:
            return grantham(x, y)
        return -1
    def _bl(r):
        x, y = r["AA_1"], r["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA:
            return blosum62(x, y)
        return 0
    out["fe_grantham"] = out.apply(_gr, axis=1).astype(float)
    out["fe_blosum62"] = out.apply(_bl, axis=1).astype(float)
    return out

FE_NEW_COLS = ["fe_aa_stopgain", "fe_aa_nonstandard", "fe_grantham", "fe_blosum62"]

def detect_log_cols(train_df):
    cand = []
    for c in NUM_COLS_BASE:
        s = pd.to_numeric(train_df[c], errors="coerce").dropna()
        if len(s) < 10: continue
        if s.min() >= 0 and s.max() > 1.0 and s.skew() > 2.0:
            cand.append(c)
    return cand

def fit_preprocessor(train_df_raw):
    tr = add_fe(train_df_raw)
    log_cols = detect_log_cols(tr)
    num_cols = NUM_COLS_BASE + FE_NEW_COLS + [f"{c}__log" for c in log_cols]
    for c in log_cols:
        tr[f"{c}__log"] = np.log1p(pd.to_numeric(tr[c], errors="coerce").clip(lower=0))
    miss = tr[base_feature_cols].isna().mean()
    flag_source = miss[miss > HIGH_MISSING_THR].index.tolist()
    median = {c: pd.to_numeric(tr[c], errors="coerce").median() for c in num_cols}
    return {
        "log_cols": log_cols, "num_cols": num_cols, "cat_cols": CAT_LIKE,
        "flag_source": flag_source, "median": median
    }

def transform_X(df_raw, pp):
    df = add_fe(df_raw)
    for c in pp["log_cols"]:
        df[f"{c}__log"] = np.log1p(pd.to_numeric(df[c], errors="coerce").clip(lower=0))
    out = pd.DataFrame(index=df.index)
    for c in pp["num_cols"]:
        out[c] = pd.to_numeric(df[c], errors="coerce").fillna(pp["median"][c]).astype(float).values
    for c in pp["cat_cols"]:
        fill = AA_UNK if c in CR.AA_COLS else "MISSING"
        out[c] = df[c].astype("object").where(~df[c].isna(), fill).astype(str).values
    for c in pp["flag_source"]:
        out[CR.get_missing_mask_col_name(c)] = df[c].isna().astype(int).values
    return out, list(pp["cat_cols"])

# MASTER ve COMBINED'a göre fit
pp_master = fit_preprocessor(master)
X_master_df, cat_cols = transform_X(master, pp_master)
y_master = master[TARGET].values

pp_combined = fit_preprocessor(combined)
X_combined_df, _ = transform_X(combined, pp_combined)
y_combined = combined[TARGET].values

# CFTR: her iki preprocessor ile transform et
X_cftr_master_df, _ = transform_X(cftr, pp_master)
X_cftr_combined_df, _ = transform_X(cftr, pp_combined)
y_cftr = cftr[TARGET].values

print(f"MASTER preprocessor (pp_master):")
print(f"  {X_master_df.shape[1]} feature, log:{len(pp_master['log_cols'])}, flag:{len(pp_master['flag_source'])}")
print(f"COMBINED preprocessor (pp_combined):")
print(f"  {X_combined_df.shape[1]} feature, log:{len(pp_combined['log_cols'])}, flag:{len(pp_combined['flag_source'])}")
print(f"CFTR: {X_cftr_master_df.shape} (master) / {X_cftr_combined_df.shape} (combined)")

MASTER preprocessor (pp_master):
  432 feature, log:0, flag:140
COMBINED preprocessor (pp_combined):
  432 feature, log:0, flag:140
CFTR: (111, 432) (master) / (111, 432) (combined)


## Cell 4: Değerlendirme Altyapısı

In [4]:
# Prior-shift düzeltmesi
def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    """Olasılığı pi_train -> pi_test için kalibre et."""
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

# Bootstrap %80/20
def _f1_pos(y, p): 
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y); prob = np.asarray(prob)
    neg = np.where(y == 0)[0]; pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0: return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED); f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {
        "mean": float(f1s.mean()), "std": float(f1s.std()),
        "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))
    }

def select_threshold_8020(y, prob):
    """Tek 8020 resampling'de F1-max threshold seç."""
    rng = np.random.RandomState(BOOT_SEED)
    yb, pb = _resample_8020(y, prob, rng)
    best, best_thr = -1.0, 0.5
    for thr in np.arange(0.05, 0.95, 0.01):
        f = _f1_pos(yb, (pb >= thr).astype(int))
        if f > best: best, best_thr = f, thr
    return float(best_thr)

# LOO-CV değerlendirmesi
def loo_metrics(y_true, oof_proba, prior_shift=False, pi_train=None):
    """LOO olasılıklarından MCC, F1, AUC hesapla."""
    if prior_shift and pi_train is not None:
        prob = adjust_prior_shift(oof_proba, pi_train, PI_TEST)
    else:
        prob = oof_proba
    thr = select_threshold_8020(y_true, prob)
    y_pred = (prob >= thr).astype(int)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.0
    f1  = _f1_pos(y_true, y_pred)
    auc = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else 0.0
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec  = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    boot = bootstrap_8020(y_true, prob, thr)
    return {
        "mcc": mcc, "f1": f1, "auc": auc, "precision": prec, "recall": rec,
        "thr": thr, "boot8020": boot,
        "y_pred": y_pred, "y_true": y_true, "prob": prob
    }

# Train metrikleri
def train_metrics_at(y_tr, p_tr, thr):
    yp = (p_tr >= thr).astype(int)
    return {
        "train_f1": _f1_pos(y_tr, yp),
        "train_mcc": matthews_corrcoef(y_tr, yp) if len(np.unique(y_tr)) > 1 else 0.0,
        "train_precision": precision_score(y_tr, yp, pos_label=1, zero_division=0),
        "train_recall": recall_score(y_tr, yp, pos_label=1, zero_division=0),
    }

print("Değerlendirme altyapısı hazır (LOO-MCC, bootstrap %80/20, prior-shift).")

Değerlendirme altyapısı hazır (LOO-MCC, bootstrap %80/20, prior-shift).


## Cell 5: Model Yardımcıları

In [5]:
def _le_encode_for_loo(X_df):
    """Kategorik sütunları label-encode et."""
    Xn = X_df.copy()
    le_maps = {}
    for c in cat_cols:
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c].astype(str))
        le_maps[c] = le
    return Xn.astype(float), le_maps

def _lgbm_classifier():
    return lgb.LGBMClassifier(**{
        **LGBM_FIXED, 
        "n_estimators": 200,
        "num_leaves": 31, 
        "learning_rate": 0.05
    })

# Encode CFTR ve train setleri
X_master_le, le_maps_master = _le_encode_for_loo(X_master_df)
X_combined_le, le_maps_combined = _le_encode_for_loo(X_combined_df)
X_cftr_master_le, _ = _le_encode_for_loo(X_cftr_master_df)
X_cftr_combined_le, _ = _le_encode_for_loo(X_cftr_combined_df)

print(f"Label-encoded: X_master {X_master_le.shape}, X_combined {X_combined_le.shape}")
print(f"Model yardımcıları hazır.")

Label-encoded: X_master (2931, 432), X_combined (3691, 432)
Model yardımcıları hazır.


## Cell 6: Ana Deney — S0_MASTER-only vs S0c_COMBINED

In [6]:
print("="*70)
print("NB19 -- CFTR Transfer Deneyi Başlıyor")
print("="*70)

results = {}

# --- S0_MASTER-only (referans, NB17 ile aynı) ---
print("\nS0_MASTER-only: MASTER (2931) ile eğit -> CFTR (111) test...")
pi_train_master = y_master.mean()
print(f"  π_train = {pi_train_master:.4f}")

m_s0 = _lgbm_classifier()
m_s0.fit(X_master_le, y_master)
p_s0_train = m_s0.predict_proba(X_master_le)[:, 1]
p_s0_cftr = m_s0.predict_proba(X_cftr_master_le)[:, 1]

# Train metriği
thr_s0_train = select_threshold_8020(y_master, p_s0_train)
train_m_s0 = train_metrics_at(y_master, p_s0_train, thr_s0_train)

# LOO metrikler (raw)
loo_s0_raw = loo_metrics(y_cftr, p_s0_cftr, prior_shift=False)

# LOO metrikler (prior-shift)
loo_s0_prior = loo_metrics(y_cftr, p_s0_cftr, prior_shift=True, pi_train=pi_train_master)

results["S0_MASTER-only"] = {
    "pi_train": pi_train_master,
    "loo_raw": loo_s0_raw,
    "loo_prior": loo_s0_prior,
    "train_metrics": train_m_s0,
    "n_train": len(y_master)
}

print(f"  LOO MCC (raw) = {loo_s0_raw['mcc']:.4f}")
print(f"  LOO MCC (prior-shift) = {loo_s0_prior['mcc']:.4f}")
print(f"  Train F1 = {train_m_s0['train_f1']:.4f}, Train MCC = {train_m_s0['train_mcc']:.4f}")
print(f"  Boot mean (prior-shift) = {loo_s0_prior['boot8020']['mean']:.4f} ± {loo_s0_prior['boot8020']['std']:.4f}")

# --- S0c_COMBINED (yeni) ---
print("\nS0c_COMBINED: MASTER+KANSER+PAH (3691) ile eğit -> CFTR (111) test...")
pi_train_combined = y_combined.mean()
print(f"  π_train = {pi_train_combined:.4f}")

m_s0c = _lgbm_classifier()
m_s0c.fit(X_combined_le, y_combined)
p_s0c_train = m_s0c.predict_proba(X_combined_le)[:, 1]
p_s0c_cftr = m_s0c.predict_proba(X_cftr_combined_le)[:, 1]

# Train metriği
thr_s0c_train = select_threshold_8020(y_combined, p_s0c_train)
train_m_s0c = train_metrics_at(y_combined, p_s0c_train, thr_s0c_train)

# LOO metrikler (raw)
loo_s0c_raw = loo_metrics(y_cftr, p_s0c_cftr, prior_shift=False)

# LOO metrikler (prior-shift)
loo_s0c_prior = loo_metrics(y_cftr, p_s0c_cftr, prior_shift=True, pi_train=pi_train_combined)

results["S0c_COMBINED"] = {
    "pi_train": pi_train_combined,
    "loo_raw": loo_s0c_raw,
    "loo_prior": loo_s0c_prior,
    "train_metrics": train_m_s0c,
    "n_train": len(y_combined)
}

print(f"  LOO MCC (raw) = {loo_s0c_raw['mcc']:.4f}")
print(f"  LOO MCC (prior-shift) = {loo_s0c_prior['mcc']:.4f}")
print(f"  Train F1 = {train_m_s0c['train_f1']:.4f}, Train MCC = {train_m_s0c['train_mcc']:.4f}")
print(f"  Boot mean (prior-shift) = {loo_s0c_prior['boot8020']['mean']:.4f} ± {loo_s0c_prior['boot8020']['std']:.4f}")

print("\n" + "="*70)

NB19 -- CFTR Transfer Deneyi Başlıyor

S0_MASTER-only: MASTER (2931) ile eğit -> CFTR (111) test...
  π_train = 0.7332


  LOO MCC (raw) = 0.6271
  LOO MCC (prior-shift) = 0.6552
  Train F1 = 0.9786, Train MCC = 0.9269
  Boot mean (prior-shift) = 0.6915 ± 0.0877

S0c_COMBINED: MASTER+KANSER+PAH (3691) ile eğit -> CFTR (111) test...
  π_train = 0.7388


  LOO MCC (raw) = 0.6436
  LOO MCC (prior-shift) = 0.6436
  Train F1 = 0.9715, Train MCC = 0.9034
  Boot mean (prior-shift) = 0.8629 ± 0.1327



## Cell 7: Sonuç Tablosu

In [7]:
rows = []
for name, res in results.items():
    loo = res["loo_prior"]; train = res["train_metrics"]; boot = loo["boot8020"]
    rows.append({
        "strategy": name,
        "n_train": res["n_train"],
        "pi_train": round(res["pi_train"], 4),
        "mcc_raw": round(res["loo_raw"]["mcc"], 4),
        "mcc_prior": round(loo["mcc"], 4),
        "f1_raw": round(res["loo_raw"]["f1"], 4),
        "f1_prior": round(loo["f1"], 4),
        "auc": round(loo["auc"], 4),
        "boot_mean": round(boot["mean"], 4),
        "boot_std": round(boot["std"], 4),
        "train_f1": round(train["train_f1"], 4),
        "train_mcc": round(train["train_mcc"], 4),
    })

results_df = pd.DataFrame(rows)
results_df.to_csv(os.path.join(RESULTS_DIR, "cftr_combined_results.csv"), index=False)

print("\n" + "="*70)
print("NB19 -- CFTR Combined Panels: S0_MASTER-only vs S0c_COMBINED")
print("="*70)
print(results_df.to_string(index=False))
print(f"\nSonuçlar kaydedildi: {os.path.join(RESULTS_DIR, 'cftr_combined_results.csv')}")


NB19 -- CFTR Combined Panels: S0_MASTER-only vs S0c_COMBINED
      strategy  n_train  pi_train  mcc_raw  mcc_prior  f1_raw  f1_prior    auc  boot_mean  boot_std  train_f1  train_mcc
S0_MASTER-only     2931    0.7332   0.6271     0.6552  0.8982    0.9186 0.9423     0.6915    0.0877    0.9786     0.9269
  S0c_COMBINED     3691    0.7388   0.6436     0.6436  0.8820    0.8820 0.9339     0.8629    0.1327    0.9715     0.9034

Sonuçlar kaydedildi: /Users/tefe/teknofest_model/teknofest_model/results/v9_cftr_combined/cftr_combined_results.csv


## Cell 8: Confusion Matrix Karşılaştırması

In [8]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, (name, res) in zip(axes, [("S0_MASTER-only", results["S0_MASTER-only"]),
                                    ("S0c_COMBINED", results["S0c_COMBINED"])]):
    loo = res["loo_prior"]
    cm = confusion_matrix(loo["y_true"], loo["y_pred"], labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["Benign", "Pathogenic"]).plot(
        ax=ax, colorbar=False, cmap="Blues"
    )
    ax.set_title(f"{name}\nMCC={loo['mcc']:.4f}, Boot-mean={res['loo_prior']['boot8020']['mean']:.4f}", 
                fontsize=10)

fig.suptitle("NB19 -- CFTR Confusion Matrix (LOO-CV + Prior-Shift)", fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "fig1_confusion_compare.png"), dpi=110, bbox_inches="tight")
plt.show()
print("Confusion matrix kaydedildi.")

Confusion matrix kaydedildi.


## Cell 9: Özet & Yorum

In [9]:
print("\n" + "="*80)
print("NB19 ÖZET & YORUM")
print("="*80)

s0_mcc = results["S0_MASTER-only"]["loo_prior"]["mcc"]
s0c_mcc = results["S0c_COMBINED"]["loo_prior"]["mcc"]
delta_mcc = s0c_mcc - s0_mcc

s0_boot = results["S0_MASTER-only"]["loo_prior"]["boot8020"]["mean"]
s0c_boot = results["S0c_COMBINED"]["loo_prior"]["boot8020"]["mean"]
delta_boot = s0c_boot - s0_boot

print(f"\nBİRİNCİL METRIK: LOO-CV MCC (prior-shift uygulanmış)")
print(f"  S0_MASTER-only : MCC = {s0_mcc:.4f}, Boot-mean = {s0_boot:.4f}")
print(f"  S0c_COMBINED   : MCC = {s0c_mcc:.4f}, Boot-mean = {s0c_boot:.4f}")
print(f"  Delta MCC      : {delta_mcc:+.4f}  (|delta| > 0.05 = anlamlı, < 0.02 = gürültü)")
print(f"  Delta Boot     : {delta_boot:+.4f}")

print(f"\nEĞİTİM HAVUZU:")
print(f"  MASTER-only : n={results['S0_MASTER-only']['n_train']}, π_train={results['S0_MASTER-only']['pi_train']:.4f}")
print(f"  COMBINED    : n={results['S0c_COMBINED']['n_train']}, π_train={results['S0c_COMBINED']['pi_train']:.4f}")

# Confusion matrix karşılaştırması
loo_s0 = results["S0_MASTER-only"]["loo_prior"]
loo_s0c = results["S0c_COMBINED"]["loo_prior"]
cm_s0 = confusion_matrix(loo_s0["y_true"], loo_s0["y_pred"], labels=[0, 1])
cm_s0c = confusion_matrix(loo_s0c["y_true"], loo_s0c["y_pred"], labels=[0, 1])

print(f"\nCONFUSION MATRIX:")
print(f"  S0_MASTER-only : TN={cm_s0[0,0]}, FP={cm_s0[0,1]}, FN={cm_s0[1,0]}, TP={cm_s0[1,1]}")
print(f"  S0c_COMBINED   : TN={cm_s0c[0,0]}, FP={cm_s0c[0,1]}, FN={cm_s0c[1,0]}, TP={cm_s0c[1,1]}")

print(f"\nVERDİKT:")
if abs(delta_mcc) < 0.02:
    verdict = "GÜRÜLTÜ - Fark istatistiksel anlamda değil (n=21 benign), COMBINED'ın avantajı yok."
elif delta_mcc > 0.05:
    verdict = "KAZANÇ - COMBINED önemli iyileştirme sağlıyor, S6 için COMBINED ile eğitim önerilir."
elif delta_mcc > 0.02:
    verdict = "ÖNERİ - Küçük iyileştirme, borderline (gözlem yeterli değilse MASTER-only tutulabilir)."
else:  # delta_mcc < 0
    verdict = "KAYIP - COMBINED performansı düşürüyor, MASTER-only tercih edilir."

print(f"  {verdict}")
print(f"\n  NOT: CFTR'de n=21 benign nedeniyle MCC farkı > 0.05 olmadıkça 'kazanan' demek uygun değildir.")
print(f"       Confusion matrix sayıları da gözlemlenmeli (FP kaymalar vb.).")

print("\n" + "="*80)


NB19 ÖZET & YORUM

BİRİNCİL METRIK: LOO-CV MCC (prior-shift uygulanmış)
  S0_MASTER-only : MCC = 0.6552, Boot-mean = 0.6915
  S0c_COMBINED   : MCC = 0.6436, Boot-mean = 0.8629
  Delta MCC      : -0.0116  (|delta| > 0.05 = anlamlı, < 0.02 = gürültü)
  Delta Boot     : +0.1713

EĞİTİM HAVUZU:
  MASTER-only : n=2931, π_train=0.7332
  COMBINED    : n=3691, π_train=0.7388

CONFUSION MATRIX:
  S0_MASTER-only : TN=18, FP=3, FN=11, TP=79
  S0c_COMBINED   : TN=21, FP=0, FN=19, TP=71

VERDİKT:
  GÜRÜLTÜ - Fark istatistiksel anlamda değil (n=21 benign), COMBINED'ın avantajı yok.

  NOT: CFTR'de n=21 benign nedeniyle MCC farkı > 0.05 olmadıkça 'kazanan' demek uygun değildir.
       Confusion matrix sayıları da gözlemlenmeli (FP kaymalar vb.).

